# Xuất bảng Parquet M1 và M2

Notebook đọc thư mục bài báo đã thu thập và xuất năm bảng Parquet chia shard. Mỗi thư mục nguồn chứa các thư mục bài báo có metadata.json.

Bảng predictions của M3 dùng tập chia và mô hình đã huấn luyện. Xem [notebook huấn luyện](train_citation_matching_kaggle.ipynb) và [hướng dẫn xuất dự đoán](../README.md#xuất-bảng-dự-đoán-parquet).

In [ ]:
#@title 1. Cài thư viện và kết nối Google Drive trên Colab
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'pandas'], check=True)

In [ ]:
#@title 2. Chọn thư mục mã nguồn
REPO_BRANCH = 'codex/phat-seed42-label-handoff'
repo_dir = Path.cwd()
if not (repo_dir / 'src/export_hf_tables.py').is_file():
    repo_dir = repo_dir / 'arXiv-Project'
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, 'https://github.com/QuangPhat101/arXiv-Project.git', str(repo_dir)], check=True)
if not (repo_dir / 'src/export_hf_tables.py').is_file():
    raise FileNotFoundError(f'Kiểm tra nhánh chứa src/export_hf_tables.py trong {repo_dir}')

In [ ]:
#@title 3. Xuất năm bảng Parquet
drive_data = Path('/content/drive/MyDrive/arxiv-m1-benchmark-20k/data')
DATA_DIRS = [drive_data if drive_data.is_dir() else repo_dir / 'data/data']
OUTPUT_DIR = Path.cwd() / 'outputs_m3/hf_m1_m2'
WORKERS = 4

# Chỉnh DATA_DIRS theo vị trí dữ liệu của nhóm. Chọn OUTPUT_DIR mới khi chạy lại.
subprocess.run([
    sys.executable, '-m', 'src.export_hf_tables',
    '--data-dir', *(str(path) for path in DATA_DIRS),
    '--output-dir', str(OUTPUT_DIR), '--workers', str(WORKERS),
], cwd=repo_dir, check=True)

In [ ]:
#@title 4. Xem số dòng và trạng thái xử lý
print((OUTPUT_DIR / 'audit.json').read_text(encoding='utf-8'))